In [ ]:
!nvidia-smi

Tue Oct 20 10:11:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |     297MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch
import time

if not torch.cuda.is_available():
    print("没有检测到 GPU，请检查运行时设置")
else:
    device = "cuda"
    print("GPU:", torch.cuda.get_device_name(0))

    a = torch.randn(3000, 3000, device=device)
    b = torch.randn(3000, 3000, device=device)

    torch.cuda.synchronize()
    start = time.time()

    result = torch.matmul(a, b)
    torch.cuda.synchronize()

    print("矩阵计算完成！")
    print("耗时:", round(time.time() - start, 4), "秒")
    print("结果形状:", result.shape)

GPU: Tesla T4
矩阵计算完成！
耗时: 0.0196 秒
结果形状: torch.Size([3000, 3000])


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
with open(
    '/content/drive/MyDrive/test.txt', 'w'
) as f:
    f.write('我的第一次 Colab GPU 实验')

In [ ]:
import os
import platform
import psutil
import subprocess

print("===== Google Colab CPU 信息 =====")

print("CPU 型号：")
subprocess.run(
    "lscpu | grep -E 'Model name|Socket|Core|Thread|CPU\\(s\\)'",
    shell=True
)

print("\nPython 可见逻辑核心数：", os.cpu_count())
print("当前进程可用逻辑核心数：",
      len(os.sched_getaffinity(0)))
print("系统总内存：",
      round(psutil.virtual_memory().total / 1024**3, 2),
      "GB")

print("\nCPU 架构：", platform.machine())

===== Google Colab CPU 信息 =====
CPU 型号：

Python 可见逻辑核心数： 2
当前进程可用逻辑核心数： 2
系统总内存： 12.67 GB

CPU 架构： x86_64


In [ ]:
!lscpu

Architecture:                x86_64
  CPU op-mode(s):            32-bit, 64-bit
  Address sizes:             46 bits physical, 48 bits virtual
  Byte Order:                Little Endian
CPU(s):                      2
  On-line CPU(s) list:       0,1
Vendor ID:                   GenuineIntel
  Model name:                Intel(R) Xeon(R) CPU @ 2.00GHz
    CPU family:              6
    Model:                   85
    Thread(s) per core:      2
    Core(s) per socket:      1
    Socket(s):               1
    Stepping:                3
    BogoMIPS:                4000.26
    Flags:                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pg
                             e mca cmov pat pse36 clflush mmx fxsr sse sse2 ss h
                             t syscall nx pdpe1gb rdtscp lm constant_tsc rep_goo
                             d nopl xtopology nonstop_tsc cpuid tsc_known_freq p
                             ni pclmulqdq ssse3 fma cx16 pcid sse4_1 sse4_2 x2ap
                   

In [ ]:
!cat /sys/fs/cgroup/cpu.max

max 100000


In [ ]:
!nvidia-smi
!nvcc --version

Tue Oct 20 10:11:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |     297MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
%pip install -q pyscf

In [ ]:
import pyscf
import sys

print("PySCF 版本:", pyscf.__version__)
print("Python 版本:", sys.version)
print("PySCF CPU 计算环境就绪！")

PySCF 版本: 2.14.0
Python 版本: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PySCF CPU 计算环境就绪！


In [ ]:
%pip install -q gpu4pyscf-cuda13x cutensor-cu13

In [ ]:
import pyscf
import cupy as cp
from gpu4pyscf.dft import rks

print("PySCF:", pyscf.__version__)
print("CuPy:", cp.__version__)

# GPU 信息
device = cp.cuda.Device(0)
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])

# 真正执行一次 CUDA GPU 运算
a = cp.arange(1000000, dtype=cp.float64)
b = cp.sum(a * a)
cp.cuda.Device(0).synchronize()

print("GPU 计算结果:", float(b.get()))
print("GPU4PySCF 导入成功！")

PySCF: 2.14.0
CuPy: 14.0.1
GPU: b'Tesla T4'
GPU 计算结果: 3.333328333335e+17
GPU4PySCF 导入成功！


In [ ]:
import time
import cupy as cp
from pyscf import gto, dft, lib

# 限制 CPU 计算线程
lib.num_threads(2)

# 建立水分子结构
mol = gto.M(
    atom="""
    O  0.0000  0.0000  0.1174
    H  0.7570  0.0000 -0.4696
    H -0.7570  0.0000 -0.4696
    """,
    basis="def2-svp",
    unit="Angstrom",
    verbose=0
)

print("开始 GPU DFT 计算...")

# 构建 DFT 模型并转移到 GPU
mf = dft.RKS(mol)
mf.xc = "B3LYP"
mf = mf.density_fit().to_gpu()

# 执行计算
start = time.perf_counter()
energy = mf.kernel()

cp.cuda.Device(0).synchronize()
elapsed = time.perf_counter() - start

print("\n===== GPU 计算结果 =====")
print("分子: H2O")
print("方法: B3LYP/def2-SVP")
print("计算对象:", type(mf).__module__)
print("总能量:", energy, "Hartree")
print("是否收敛:", mf.converged)
print("计算时间:", round(elapsed, 3), "秒")

开始 GPU DFT 计算...


RuntimeError: failed in block_diag kernel

In [ ]:
!nvidia-smi
!nvcc --version
!python -m pip list | grep -Ei "pyscf|cupy|cutensor"

Tue Oct 20 10:16:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             32W /   70W |     321MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from pyscf import gto, lib
from gpu4pyscf.dft import rks
import cupy as cp
import traceback

lib.num_threads(2)

mol = gto.M(
    atom="H 0 0 0; H 0 0 0.74",
    basis="sto-3g",
    verbose=4
)

mf = rks.RKS(mol, xc="PBE").density_fit()

# 尝试使用单电子哈密顿量生成初始密度
mf.init_guess = "1e"

print("计算对象:", type(mf))
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])

try:
    energy = mf.kernel()
    cp.cuda.Device(0).synchronize()

    print("总能量:", energy, "Hartree")
    print("是否收敛:", mf.converged)

except Exception:
    traceback.print_exc()

System: uname_result(system='Linux', node='34341b59784d', release='6.6.122+', version='#1 SMP Thu Apr 30 18:17:14 UTC 2026', machine='x86_64')  Threads 2
Python 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
numpy 2.1.3  scipy 1.16.3  h5py 3.14.0
Date: Tue Oct 20 10:17:16 2026
PySCF version 2.14.0
PySCF path  /usr/local/lib/python3.13/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path /usr/local/lib/python3.13/dist-packages/nvidia/cu13
    CUDA Build Version 13000
    CUDA Driver Version 13000
    CUDA Runtime Version 13000
CUDA toolkit
    cuSolver (12, 0, 4)
    cuBLAS 130100
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 1
GPU4PySCF 1.8.1
GPU4PySCF path  /usr/local/lib/python3.13/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 2
[INPUT] num. electrons = 2
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INP

Traceback (most recent call last):
  File "/tmp/ipykernel_673/1812433266.py", line 23, in <cell line: 0>
    energy = mf.kernel()
  File "/usr/local/lib/python3.13/dist-packages/gpu4pyscf/scf/hf.py", line 390, in scf
    _kernel(mf, mf.conv_tol, mf.conv_tol_grad,
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            dm0=dm0, callback=mf.callback,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            conv_check=mf.conv_check, **kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/gpu4pyscf/scf/hf.py", line 230, in _kernel
    dm0 = mf.get_init_guess(mol, mf.init_guess)
  File "/usr/local/lib/python3.13/dist-packages/pyscf/scf/hf.py", line 2024, in get_init_guess
    dm = self.init_guess_by_1e(mol)
  File "/usr/local/lib/python3.13/dist-packages/gpu4pyscf/scf/hf.py", line 625, in fn_init_guess
    dm = fn(mf, mol)
  File "/usr/local/lib/python3.13/dist-packages/pyscf/scf/hf.py", line 1977, in init_guess_by_1e
    h1e = self.get_hco

In [ ]:
import cupy as cp
import subprocess

print("===== GPU 底层诊断 =====")

# 检查 GPU
dev = cp.cuda.Device(0)
print("GPU 计算能力:", dev.compute_capability)

# 测试 CuPy CUDA 内核
try:
    x = cp.arange(100000, dtype=cp.float64)
    y = cp.sum(x * x)
    print("CuPy CUDA 计算成功:", float(y.get()))
except Exception as e:
    print("CuPy CUDA 计算失败:", repr(e))

# 检查 CUDA 库
print("\n===== CUDA 库信息 =====")
cp.show_config()

# 检查 Python 依赖
print("\n===== Python 依赖检查 =====")
subprocess.run(
    ["python", "-m", "pip", "check"]
)

===== GPU 底层诊断 =====
GPU 计算能力: 75
CuPy CUDA 计算成功: 333328333350000.0

===== CUDA 库信息 =====
OS                           : Linux-6.6.122+-x86_64-with-glibc2.39
Python Version               : 3.13.15
CuPy Version                 : 14.0.1
CuPy Platform                : NVIDIA CUDA
NumPy Version                : 2.1.3
SciPy Version                : 1.16.3
Cython Build Version         : 3.1.4
Cython Runtime Version       : 3.0.12
CUDA Root                    : None
NVCC PATH                    : None
CUDA Build Version           : 13000
CUDA Driver Version          : 13000
CUDA Runtime Version         : 13000 (linked to CuPy) / 13000 (locally installed)
CUDA Extra Include Dirs      : ['/usr/local/lib/python3.13/dist-packages/nvidia/cu13/include']
cuBLAS Version               : (available)
cuFFT Version                : 12000
cuRAND Version               : 10400
cuSOLVER Version             : (12, 0, 4)
cuSPARSE Version             : (available)
NVRTC Version                : (13, 0)
Thrust V

CompletedProcess(args=['python', '-m', 'pip', 'check'], returncode=1)

In [ ]:
import subprocess

print("===== 1. 检查依赖冲突 =====")
subprocess.run(
    ["python", "-m", "pip", "check"],
    check=False
)

print("\n===== 2. 检查相关软件包 =====")
subprocess.run(
    "python -m pip list | grep -Ei "
    "'pyscf|cupy|cutensor|cuda|nvidia'",
    shell=True,
    check=False
)

===== 1. 检查依赖冲突 =====

===== 2. 检查相关软件包 =====


CompletedProcess(args="python -m pip list | grep -Ei 'pyscf|cupy|cutensor|cuda|nvidia'", returncode=0)

In [ ]:
import sys
import subprocess
import importlib.metadata as md

print("========== 1. Python 环境 ==========")
print("Python 路径:", sys.executable)

print("\n========== 2. 完整依赖检查 ==========")

result = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    capture_output=True,
    text=True
)

print("退出状态码:", result.returncode)
print("标准输出:")
print(result.stdout or "(无)")
print("错误输出:")
print(result.stderr or "(无)")

print("\n========== 3. GPU 相关软件包 ==========")

keywords = (
    "pyscf", "cupy", "cutensor",
    "cuda", "nvidia"
)

packages = sorted(
    md.distributions(),
    key=lambda d: d.metadata.get("Name", "").lower()
)

for package in packages:
    name = package.metadata.get("Name", "")
    if any(k in name.lower() for k in keywords):
        print(f"{name:42s} {package.version}")

print("\n========== 检查完成 ==========")

========== 1. Python 环境 ==========
Python 路径: /usr/bin/python3

========== 2. 完整依赖检查 ==========
退出状态码: 1
标准输出:
ipython 7.34.0 requires jedi, which is not installed.

错误输出:
(无)

========== 3. GPU 相关软件包 ==========
cuda-bindings                              13.0.3
cuda-core                                  0.7.0
cuda-pathfinder                            1.8.3
cuda-python                                13.0.3
cuda-toolkit                               13.0.2
cupy-cuda13x                               14.0.1
cutensor-cu13                              2.8.1
dask-cuda                                  26.6.0
gpu4pyscf-cuda13x                          1.8.1
gpu4pyscf-libxc-cuda13x                    0.8.1
jax-cuda13-pjrt                            0.11.1
jax-cuda13-plugin                          0.11.1
numba-cuda                                 0.28.2
nvidia-cublas                              13.1.0.3
nvidia-cublas-cu12                         12.9.2.10
nvidia-cuda-cccl                      

In [ ]:
%pip uninstall -y \
    gpu4pyscf-cuda13x \
    gpu4pyscf-libxc-cuda13x \
    cupy-cuda13x \
    cutensor-cu13

Found existing installation: cupy-cuda13x 14.0.1
Uninstalling cupy-cuda13x-14.0.1:
  Successfully uninstalled cupy-cuda13x-14.0.1


In [ ]:
%pip install --no-cache-dir \
    "pyscf==2.14.0" \
    "gpu4pyscf-cuda12x==1.8.1" \
    "gpu4pyscf-libxc-cuda12x==0.8.1" \
    "cupy-cuda12x==13.4.1" \
    "cutensor-cu12==2.2.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.2/392.2 kB 17.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 236.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.7/151.7 MB 192.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 204.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 223.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 213.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 209.1 MB/s eta 0:00:00
  Created wheel for geometric: filename=geometric-1.1.1-py3-none-any.whl size=408351 sha256=f6d02217a68cfbad34d1202b991b8e4de5f67df9f65c7507be247e4561c08c56
  Stored in directory: /tmp/pip-ephem-wheel-cache-hhoy6dzi/wheels/82/fe/a4/91e2803e6eafb64e10e66e04ecae5682c90f7c55bc28201573
Successfully built geometric


In [ ]:
import cupy as cp
import pyscf
import importlib.metadata as md

print("===== GPU 计算化学环境 =====")

for name in [
    "pyscf",
    "gpu4pyscf-cuda12x",
    "cupy-cuda12x",
    "cutensor-cu12"
]:
    print(name, md.version(name))

print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])
print("CUDA Runtime:", cp.cuda.runtime.runtimeGetVersion())

# 实际测试 GPU
x = cp.arange(10000, dtype=cp.float64)
print("GPU 运算结果:", float(cp.sum(x).get()))

===== GPU 计算化学环境 =====
pyscf 2.14.0
gpu4pyscf-cuda12x 1.8.1
cupy-cuda12x 13.4.1
cutensor-cu12 2.2.0
GPU: b'Tesla T4'
CUDA Runtime: 12080


RuntimeError: CuPy failed to load libnvrtc.so.12: OSError: libnvrtc.so.12: cannot open shared object file: No such file or directory

In [ ]:
%pip install -q "nvidia-cuda-nvrtc-cu12==12.8.93"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 8.3 MB/s eta 0:00:00


In [ ]:
import ctypes
import site
from pathlib import Path

print("===== 检查 CUDA 12 NVRTC =====")

paths = []

for folder in site.getsitepackages():
    paths.extend(
        Path(folder).glob(
            "nvidia/cuda_nvrtc/lib/libnvrtc.so.12"
        )
    )

print("找到的 NVRTC 库:", paths)

if not paths:
    print("未找到 NVRTC，请检查安装结果")
else:
    ctypes.CDLL(
        str(paths[0]),
        mode=ctypes.RTLD_GLOBAL
    )
    print("NVRTC 加载成功！")

===== 检查 CUDA 12 NVRTC =====
找到的 NVRTC 库: [PosixPath('/usr/local/lib/python3.13/dist-packages/nvidia/cuda_nvrtc/lib/libnvrtc.so.12')]
NVRTC 加载成功！


In [ ]:
import cupy as cp

print("===== CUDA 12 GPU 测试 =====")

print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])
print("CUDA Runtime:", cp.cuda.runtime.runtimeGetVersion())

x = cp.arange(10000, dtype=cp.float64)
result = cp.sum(x)

cp.cuda.Device(0).synchronize()

print("GPU 计算结果:", float(result.get()))
print("CuPy GPU 测试成功！")

===== CUDA 12 GPU 测试 =====
GPU: b'Tesla T4'
CUDA Runtime: 13000
GPU 计算结果: 49995000.0
CuPy GPU 测试成功！


In [ ]:
import time
import traceback
import cupy as cp

from pyscf import gto, dft, lib
from gpu4pyscf.dft import rks

lib.num_threads(2)

print("===== GPU4PySCF 计算验证 =====")

# 1. 建立 H2 分子
mol = gto.M(
    atom="""
    H 0.0 0.0 0.0
    H 0.0 0.0 0.74
    """,
    basis="sto-3g",
    unit="Angstrom",
    verbose=0
)

# 2. CPU 参考计算
print("\n【1】CPU PBE-DFT")

cpu_mf = dft.RKS(mol).density_fit()
cpu_mf.xc = "PBE"
cpu_mf.init_guess = "1e"
cpu_mf.grids.level = 3

start = time.perf_counter()
cpu_energy = cpu_mf.kernel()
cpu_time = time.perf_counter() - start

print("CPU 能量:", cpu_energy, "Hartree")
print("CPU 收敛:", cpu_mf.converged)
print("CPU 耗时:", round(cpu_time, 3), "秒")

# 3. GPU 计算
print("\n【2】GPU PBE-DFT")

try:
    gpu_mf = rks.RKS(mol, xc="PBE").density_fit()
    gpu_mf.init_guess = "1e"
    gpu_mf.grids.level = 3

    print("GPU 计算对象:", type(gpu_mf))

    start = time.perf_counter()
    gpu_energy = gpu_mf.kernel()

    cp.cuda.Device(0).synchronize()
    gpu_time = time.perf_counter() - start

    print("GPU 能量:", gpu_energy, "Hartree")
    print("GPU 收敛:", gpu_mf.converged)
    print("GPU 耗时:", round(gpu_time, 3), "秒")

    if cpu_mf.converged and gpu_mf.converged:
        difference = abs(float(cpu_energy) - float(gpu_energy))
        print("\nCPU/GPU 能量差:", difference, "Hartree")

    print("\nGPU4PySCF 计算流程执行成功！")

except Exception:
    print("\nGPU4PySCF 计算失败，错误如下：")
    traceback.print_exc()

ModuleNotFoundError: No module named 'pyscf'

In [ ]:
%pip install -q "pyscf==2.14.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.2 MB/s eta 0:00:00


In [ ]:
import pyscf

print("PySCF 版本:", pyscf.__version__)
print("PySCF 安装成功！")

PySCF 版本: 2.14.0
PySCF 安装成功！


In [ ]:
import importlib.util
import importlib.metadata as md

packages = {
    "PySCF": "pyscf",
    "GPU4PySCF": "gpu4pyscf",
    "CuPy": "cupy"
}

print("===== 计算化学环境检查 =====")

for name, module in packages.items():
    found = importlib.util.find_spec(module)
    print(f"{name}: {'已安装' if found else '未安装'}")

print("\n===== GPU 软件版本 =====")

for name in [
    "gpu4pyscf-cuda12x",
    "gpu4pyscf-cuda13x",
    "cupy-cuda12x",
    "cupy-cuda13x",
    "cutensor-cu12",
    "cutensor-cu13",
    "nvidia-cuda-nvrtc-cu12"
]:
    try:
        print(name, md.version(name))
    except md.PackageNotFoundError:
        pass

===== 计算化学环境检查 =====
PySCF: 已安装
GPU4PySCF: 未安装
CuPy: 已安装

===== GPU 软件版本 =====
cupy-cuda13x 14.0.1
nvidia-cuda-nvrtc-cu12 12.8.93


In [ ]:
!python3 -m venv /content/pyscf_gpu12

Error: Command '['/content/pyscf_gpu12/bin/python3', '-m', 'ensurepip', '--upgrade', '--default-pip']' returned non-zero exit status 1.


In [ ]:
!python -m pip install -q virtualenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.1/472.1 kB 42.1 MB/s eta 0:00:00


In [ ]:
!python -m virtualenv --clear /content/pyscf_gpu12

created virtual environment CPython3.13.15.final.0-64-x86_64 in 462ms
  creator CPython3Posix(dest=/content/pyscf_gpu12, clear=True, no_vcs_ignore=False, venv_redirect=None, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.2.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator,XonshActivator


In [ ]:
!/content/pyscf_gpu12/bin/python --version
!/content/pyscf_gpu12/bin/python -m pip --version

Python 3.13.15
pip 26.2.1 from /content/pyscf_gpu12/lib/python3.13/site-packages/pip (python 3.13)


In [ ]:
!/content/pyscf_gpu12/bin/python -m pip install --no-cache-dir \
    "pyscf==2.14.0" \
    "gpu4pyscf-cuda12x==1.8.1" \
    "gpu4pyscf-libxc-cuda12x==0.8.1" \
    "cupy-cuda12x==13.4.1" \
    "cutensor-cu12==2.2.0" \
    "nvidia-cuda-runtime-cu12>=12.8,<12.9" \
    "nvidia-cuda-nvrtc-cu12>=12.8,<12.9"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 310.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.7/151.7 MB 173.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 62.7 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 MB 65.2 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.3/202.3 MB 65.0 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 37.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 55.8 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 69.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 70.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 76.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 75.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━

In [ ]:
import os
import subprocess
from pathlib import Path

# 独立 Python 环境
venv = Path("/content/pyscf_gpu12")
python = str(venv / "bin/python")

site = next((venv / "lib").glob("python*/site-packages"))

# 优先使用独立环境中的 CUDA 12 库
libdirs = [
    site / "nvidia/cuda_runtime/lib",
    site / "nvidia/cuda_nvrtc/lib",
    site / "nvidia/cublas/lib",
    site / "nvidia/cusolver/lib",
    site / "nvidia/cusparse/lib",
    site / "nvidia/nvjitlink/lib",
    site / "cutensor/lib",
]

env = os.environ.copy()
paths = [str(p) for p in libdirs if p.is_dir()]

if env.get("LD_LIBRARY_PATH"):
    paths.append(env["LD_LIBRARY_PATH"])

env["LD_LIBRARY_PATH"] = ":".join(paths)

# 在独立环境执行测试
test = """
import pyscf
import cupy as cp
from gpu4pyscf.dft import rks

print("PySCF:", pyscf.__version__)
print("CuPy:", cp.__version__)
print("GPU:", cp.cuda.runtime.getDeviceProperties(0)["name"])
print("CUDA Runtime:", cp.cuda.runtime.runtimeGetVersion())

x = cp.arange(10000, dtype=cp.float64)
y = cp.sum(x)
cp.cuda.Device(0).synchronize()

print("GPU 计算结果:", float(y.get()))
print("GPU 基础测试成功！")
"""

result = subprocess.run(
    [python, "-c", test],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("退出状态码:", result.returncode)

PySCF: 2.14.0
CuPy: 13.4.1
GPU: b'Tesla T4'
CUDA Runtime: 12080
GPU 计算结果: 49995000.0
GPU 基础测试成功！

/content/pyscf_gpu12/lib/python3.13/site-packages/cupy/_environment.py:370: UserWarning: CuPy failed to preload library (/content/pyscf_gpu12/lib/python3.13/site-packages/cutensor/lib/libcutensor.so.2): OSError (libcublasLt.so.12: cannot open shared object file: No such file or directory)
  warnings.warn(msg)
/content/pyscf_gpu12/lib/python3.13/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')

退出状态码: 0


In [ ]:
gpu_test = """
from pyscf import gto
from gpu4pyscf.dft import rks
import cupy as cp
import time

mol = gto.M(
    atom="H 0 0 0; H 0 0 0.74",
    basis="sto-3g",
    unit="Angstrom",
    verbose=0
)

mf = rks.RKS(mol, xc="PBE").density_fit()
mf.init_guess = "1e"

print("计算对象:", type(mf), flush=True)
print("开始 GPU-DFT 计算...", flush=True)

start = time.perf_counter()
energy = mf.kernel()

cp.cuda.Device(0).synchronize()

print("总能量:", energy, "Hartree")
print("SCF 收敛:", mf.converged)
print("耗时:", round(time.perf_counter()-start, 3), "秒")
"""

result = subprocess.run(
    [python, "-u", "-c", gpu_test],
    env=env,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
print("退出状态码:", result.returncode)

计算对象: <class 'gpu4pyscf.df.df_jk.DFRKS'>
开始 GPU-DFT 计算...

/content/pyscf_gpu12/lib/python3.13/site-packages/cupy/_environment.py:370: UserWarning: CuPy failed to preload library (/content/pyscf_gpu12/lib/python3.13/site-packages/cutensor/lib/libcutensor.so.2): OSError (libcublasLt.so.12: cannot open shared object file: No such file or directory)
  warnings.warn(msg)
/content/pyscf_gpu12/lib/python3.13/site-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')
Traceback (most recent call last):
  File "<string>", line 21, in <module>
    energy = mf.kernel()
  File "/content/pyscf_gpu12/lib/python3.13/site-packages/gpu4pyscf/scf/hf.py", line 390, in scf
    _kernel(mf, mf.conv_tol, mf.conv_tol_grad,
    ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            dm0=dm0, callback=mf.callback,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            conv_check=mf.conv_check

In [ ]:
!/content/pyscf_gpu12/bin/python -m pip install \
    "nvidia-cublas-cu12==12.8.4.1" \
    "nvidia-cusolver-cu12==11.7.3.90" \
    "nvidia-cusparse-cu12==12.5.8.93" \
    "nvidia-nvjitlink-cu12==12.8.93"

/bin/bash: line 1: /content/pyscf_gpu12/bin/python: No such file or directory


In [ ]:
import sys
import subprocess
from pathlib import Path

venv = Path("/content/pyscf_gpu12")
python = venv / "bin/python"

print("当前 Python:", sys.executable)
print("旧虚拟环境存在:", python.exists())

if not python.exists():
    print("正在重新创建虚拟环境...")

    subprocess.run(
        [sys.executable, "-m", "pip",
         "install", "-q", "virtualenv"],
        check=True
    )

    subprocess.run(
        [sys.executable, "-m", "virtualenv",
         str(venv)],
        check=True
    )

print("\n验证虚拟环境：")
subprocess.run(
    [str(python), "-m", "pip", "--version"],
    check=True
)

当前 Python: /usr/bin/python3
旧虚拟环境存在: False
正在重新创建虚拟环境...

验证虚拟环境：


CompletedProcess(args=['/content/pyscf_gpu12/bin/python', '-m', 'pip', '--version'], returncode=0)

In [ ]:
import sys
import subprocess
from pathlib import Path

VENV = Path("/content/pyscf_gpu12")
PYTHON = VENV / "bin/python"

# 环境不存在时自动创建
if not PYTHON.exists():
    subprocess.run(
        [sys.executable, "-m", "pip",
         "install", "-q", "virtualenv"],
        check=True
    )
    subprocess.run(
        [sys.executable, "-m", "virtualenv", str(VENV)],
        check=True
    )

packages = [
    "pyscf==2.14.0",
    "gpu4pyscf-cuda12x==1.8.1",
    "gpu4pyscf-libxc-cuda12x==0.8.1",
    "cupy-cuda12x==13.4.1",
    "cutensor-cu12==2.2.0",
    "nvidia-cuda-runtime-cu12==12.8.90",
    "nvidia-cuda-nvrtc-cu12==12.8.93",
    "nvidia-cublas-cu12==12.8.4.1",
    "nvidia-cusolver-cu12==11.7.3.90",
    "nvidia-cusparse-cu12==12.5.8.93",
    "nvidia-nvjitlink-cu12==12.8.93",
]

print("正在检查并安装 CUDA 12 环境...", flush=True)

subprocess.run(
    [str(PYTHON), "-m", "pip", "install",
     "--disable-pip-version-check", *packages],
    check=True
)

print("\n===== 检查依赖 =====", flush=True)
subprocess.run(
    [str(PYTHON), "-m", "pip", "check"],
    check=True
)

print("\n环境初始化完成！")

正在检查并安装 CUDA 12 环境...

===== 检查依赖 =====

环境初始化完成！


In [ ]:
import os
import subprocess
from pathlib import Path

venv = Path("/content/pyscf_gpu12")
python = str(venv / "bin/python")
site = next((venv / "lib").glob("python*/site-packages"))

# 优先加载虚拟环境中的 CUDA 12 库
folders = [
    "nvidia/nvjitlink/lib",
    "nvidia/cuda_runtime/lib",
    "nvidia/cuda_nvrtc/lib",
    "nvidia/cublas/lib",
    "nvidia/cusparse/lib",
    "nvidia/cusolver/lib",
    "cutensor/lib",
]

env = os.environ.copy()
env["LD_LIBRARY_PATH"] = ":".join(
    str(site / p)
    for p in folders
    if (site / p).is_dir()
)

test_code = """
import cupy as cp

print("GPU:",
      cp.cuda.runtime.getDeviceProperties(0)["name"])
print("CuPy 版本:", cp.__version__)
print("CUDA Runtime:",
      cp.cuda.runtime.runtimeGetVersion())

# 测试 CUDA 基础计算
x = cp.arange(10000, dtype=cp.float64)
print("CuPy 求和:", float(cp.sum(x).get()))

# 测试 CuPy 本征值计算
H = cp.array(
    [[2.0, 0.1], [0.1, 3.0]],
    dtype=cp.float64
)

w, v = cp.linalg.eigh(H)
print("CuPy 本征值:", w.get())

# 测试 GPU4PySCF 专用求解内核
from gpu4pyscf.lib.cusolver import eigh

S = cp.eye(2, dtype=cp.float64)
w2, v2 = eigh(H, S)

cp.cuda.Device(0).synchronize()

print("GPU4PySCF 本征值:", cp.asnumpy(w2))
print("GPU4PySCF 底层测试通过！")
"""

result = subprocess.run(
    [python, "-u", "-c", test_code],
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    timeout=120
)

print(result.stdout)
print("退出状态码:", result.returncode)

Traceback (most recent call last):
  File "<string>", line 5, in <module>
    cp.cuda.runtime.getDeviceProperties(0)["name"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "cupy_backends/cuda/api/runtime.pyx", line 211, in cupy_backends.cuda.api.runtime.getDeviceProperties
  File "cupy_backends/cuda/api/runtime.pyx", line 214, in cupy_backends.cuda.api.runtime.getDeviceProperties
  File "cupy_backends/cuda/api/runtime.pyx", line 146, in cupy_backends.cuda.api.runtime.check_status
cupy_backends.cuda.api.runtime.CUDARuntimeError: cudaErrorInsufficientDriver: CUDA driver version is insufficient for CUDA runtime version

退出状态码: 1


In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. 加载经典鸢尾花数据集
iris = load_iris()
X, y = iris.data, iris.target

# 2. 划分训练集和测试集（80% 训练，20% 测试）
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. 初始化并训练逻辑回归模型
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# 4. 在测试集上进行预测
y_pred = model.predict(X_test)

# 5. 评估模型性能
accuracy = accuracy_score(y_test, y_pred)
print("===== 机器学习模型训练演示 =====")
print(f"模型测试集准确率: {accuracy:.4f} ({accuracy * 100:.2f}%)")
print("\n分类报告:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

===== 机器学习模型训练演示 =====
模型测试集准确率: 1.0000 (100.00%)

分类报告:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30

